In [1]:
#!/usr/bin/env python
import argparse
from collections import defaultdict
import csv
from pathlib import Path
import networkx as nx
import numpy as np

def import_nw(nw_path):
	G = nx.Graph()
	with open(nw_path) as nw_file:
		csv_reader = csv.reader(nw_file)
		for row_index, row in enumerate(csv_reader):
			if row_index == 0:
				try:
					n1_index = int(row.index("node1"))
					n2_index = int(row.index("node2"))
				except ValueError:
					raise Exception("The input table must contain columns titled \"node1\" and \"node2\"")
			else:
				n1 = row[n1_index]
				n2 = row[n2_index]
				G.add_edge(n1, n2)
	return G

def read_node_type_map(type_map_path):
	type_map = defaultdict(list)
	with open(type_map_path) as type_map_file:
		csv_reader = csv.reader(type_map_file)
		for row in csv_reader:
			(node_name, node_type) = row
			type_map[node_type].append(node_name)
	return type_map

def subgraph_intersection(subgraph1, subgraph2):
	return np.intersect1d(subgraph1, subgraph2)

def get_type_subgraph(graph, type_map, type):
	if type not in type_map:
		raise Exception(f"Type {type} not in type map")
	return subgraph_intersection(type_map[type], graph)

def connected_component_subgraphs(network):
	return [network.subgraph(component) for component in nx.connected_components(network)]

# Returns a dictionary: node name -> BiBC
def bibc(G, nodes_0, nodes_1, normalized):
	bibcs = {n: 0.0 for n in G}
	for s in nodes_0:
		for t in nodes_1:
			# betweenness centrality does not count the endpoints (v not in s,t)
			paths_st = [x for x in list(nx.all_shortest_paths(G,s,t)) if len(x) > 2]
			n_paths = len(paths_st)
			for path in paths_st:
				for n in path[1:-1]: # Exclude endpoints
					bibcs[n] += 1 / n_paths

	if normalized:
		possible_paths_t0 = (len(nodes_0) - 1) * len(nodes_1)
		possible_paths_t1 = len(nodes_0) * (len(nodes_1) - 1)
		possible_paths_other = len(nodes_0) * len(nodes_1)
		for n in bibcs:
			if n in nodes_0:
				bibcs[n] /= possible_paths_t0
			elif n in nodes_1:
				bibcs[n] /= possible_paths_t1
			else:
				bibcs[n] /= possible_paths_other

	return bibcs

def write_bibc(bibc_map, file_path):
	with open(file_path, "w") as bibc_file:
		writer = csv.writer(bibc_file)
		writer.writerow(["node", "BiBC"])
		for n, bibc in bibc_map.items():
			writer.writerow([n, bibc])

parser = argparse.ArgumentParser(description="Computes the BiBC of every node in a network, relative to two types of nodes (i.e. subnetworks).")
parser.add_argument("--network", required=True, help="The path of a CSV file listing the edges in the network. It must include columns titled \"node1\" and \"node2\" that indicate the endpoints of each edge.")
parser.add_argument("--type_map", required=True, help="The path of a CSV file mapping nodes to types/classes. It must not have a header. The first column must contain node names and the second column must contain the type of the corresponding node.")
parser.add_argument("--type1", required=True, help="The node type to use for one end of the BiBC calculation.")
parser.add_argument("--type2", required=True, help="The node type to use for the other end of the BiBC calculation.")
parser.add_argument("--normalized", action="store_true", default=False, help="Normalize BiBC to be between 0 and 1. A normalized BiBC of 1 means the node appears on all shortest paths between the two node types; a value of 0 means it appears on none of them.")
parser.add_argument("--output", required=True, help="The path of a CSV file that will be produced containing the names of nodes (the first column) and their BiBCs (the second column).")
args = parser.parse_args()

network_path = Path(args.network)
if not network_path.exists:
	raise Exception(f"Specified network file {network_path} does not exist.")
	
output_path = Path(args.output)

G = import_nw(network_path)
gc = max(connected_component_subgraphs(G), key=len)

type_map = read_node_type_map(args.type_map)
type1_subgraph = get_type_subgraph(gc, type_map, args.type1)
type2_subgraph = get_type_subgraph(gc, type_map, args.type2)

bibc_map = bibc(gc, type1_subgraph, type2_subgraph, args.normalized)
write_bibc(bibc_map, output_path)

usage: ipykernel_launcher.py [-h] --network NETWORK --type_map TYPE_MAP
                             --type1 TYPE1 --type2 TYPE2 [--normalized]
                             --output OUTPUT
ipykernel_launcher.py: error: the following arguments are required: --network, --type_map, --type1, --type2, --output


SystemExit: 2

/opt/anaconda3/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [11]:
import pandas as pd


# Feci-Feci
feci_feci = pd.read_csv("Feci-Feci.csv")
feci_feci = feci_feci.apply(lambda c: c.str.strip() if c.dtype == "object" else c)
pair_filt = pd.read_csv("../Feci-CPX/feci-feci_node_pairs.csv")
pair_filt = pair_filt.apply(lambda c: c.str.strip() if c.dtype == "object" else c)
allowed_pairs = set(tuple(sorted((r.node1, r.node2))) for i, r in pair_filt.iterrows())
feci_feci = feci_feci[feci_feci.apply(lambda r: tuple(sorted((r["Metabolite 1"], r["Metabolite 2"]))) in allowed_pairs,axis=1)]
feci_feci = feci_feci[feci_feci['FDR'].notna()]
feci_feci = feci_feci[feci_feci['FDR'] <= 0.05]
feci_feci = feci_feci[feci_feci[["VECPAC p-values", "DSS p-values", "LPS p-values"]].max(axis=1, skipna=True) <= 0.2]
feci_feci = feci_feci[['Metabolite 1', 'Metabolite 2']]
feci_feci = feci_feci.rename(columns={'Metabolite 1': 'node1','Metabolite 2': 'node2'})
feci_feci[['node1', 'node2']] = feci_feci[['node1', 'node2']].astype(str) + '-F'
feci_feci.to_csv("Feci-Feci_updated.csv", index=False)
print("Feci-FECI",len(feci_feci))


# TESTING PURPOSES ONLY ------------------------------------------------------------------------
# Feci-CPX
feci_pls = pd.read_csv("Feci-CPX.csv")
feci_pls = feci_pls.apply(lambda c: c.str.strip() if c.dtype == "object" else c)
feci_pls = feci_pls[feci_pls['Pooled FDR'].notna()]
feci_pls = feci_pls[feci_pls['Pooled FDR'] <= 0.1]
feci_pls = feci_pls[['Gene', 'Metabolite']]
feci_pls = feci_pls.rename(columns={'Gene': 'node1','Metabolite': 'node2'})
feci_pls['node1'] = feci_pls['node1'].str.replace(r'-P$', '', regex=True)
feci_pls = feci_pls.apply(lambda c: c.str.strip() if c.dtype == "object" else c)
feci_pls['node1'] = feci_pls['node1'].astype(str) + '-C'
feci_pls['node2'] = feci_pls['node2'].astype(str) + '-F'
feci_pls.to_csv("Feci-CPX_updated.csv", index=False)
print("Feci-CPX",len(feci_pls))
# TESTING PURPOSES ONLY ------------------------------------------------------------------------


# Feci-PLS
feci_pls = pd.read_csv("Feci-PLS.csv")
feci_pls = feci_pls.apply(lambda c: c.str.strip() if c.dtype == "object" else c)
feci_pls = feci_pls[feci_pls['Pooled FDR'].notna()]
feci_pls = feci_pls[feci_pls['Pooled FDR'] <= 0.1]
feci_pls = feci_pls[['Gene', 'Metabolite']]
feci_pls = feci_pls.rename(columns={'Gene': 'node1','Metabolite': 'node2'})
feci_pls['node1'] = feci_pls['node1'].str.replace(r'-Feci$', '', regex=True)
feci_pls['node2'] = feci_pls['node2'].str.replace(r'-PLS$', '', regex=True)
feci_pls = feci_pls.apply(lambda c: c.str.strip() if c.dtype == "object" else c)
feci_pls['node1'] = feci_pls['node1'].astype(str) + '-F'
feci_pls['node2'] = feci_pls['node2'].astype(str) + '-P'
feci_pls.to_csv("Feci-PLS_updated.csv", index=False)
print("Feci-PLS",len(feci_pls))

# PLS-PLS
pls_pls = pd.read_csv("PLS-PLS.csv")
pls_pls = pls_pls.apply(lambda c: c.str.strip() if c.dtype == "object" else c)
pair_filt = pd.read_csv("../PLS-CPX/pls-pls_node_pairs.csv")
pair_filt = pair_filt.apply(lambda c: c.str.strip() if c.dtype == "object" else c)
allowed_pairs = set(tuple(sorted((r.node1, r.node2))) for i, r in pair_filt.iterrows())
pls_pls = pls_pls[pls_pls.apply(lambda r: tuple(sorted((r["Metabolite 1"], r["Metabolite 2"]))) in allowed_pairs,axis=1)]
pls_pls = pls_pls[pls_pls['FDR'].notna()]
pls_pls = pls_pls[pls_pls['FDR'] <= 0.05]
pls_pls = pls_pls[pls_pls[["VECPAC p-values", "DSS p-values", "LPS p-values"]].max(axis=1, skipna=True) <= 0.2]
pls_pls = pls_pls[['Metabolite 1', 'Metabolite 2']]
pls_pls = pls_pls.rename(columns={'Metabolite 1': 'node1','Metabolite 2': 'node2'})
pls_pls[['node1', 'node2']] = pls_pls[['node1', 'node2']].astype(str) + '-P'
pls_pls.to_csv("PLS-PLS_updated.csv", index=False)
print("PLS-PLS",len(pls_pls))



# PLS-CPX
pls_cpx = pd.read_csv("PLS-CPX.csv")
pls_cpx = pls_cpx.rename(columns={'Gene': 'node1','Metabolite': 'node2'})
pls_cpx['node1'] = pls_cpx['node1'].str.replace(r'-P$', '', regex=True)
pls_cpx = pls_cpx.apply(lambda c: c.str.strip() if c.dtype == "object" else c)
pair_filt = pd.read_csv("../PLS-CPX/cpx-pls_node_pairs.csv")
pair_filt = pair_filt.apply(lambda c: c.str.strip() if c.dtype == "object" else c)
allowed_pairs = set(tuple(sorted((r.node1, r.node2))) for i, r in pair_filt.iterrows())
pls_cpx = pls_cpx[pls_cpx['Pooled FDR'].notna()]
pls_cpx = pls_cpx[pls_cpx['Pooled FDR'] <= 0.1]
pls_cpx = pls_cpx[['node1', 'node2']]
pls_cpx = pls_cpx[pls_cpx.apply(lambda r: tuple(sorted((r["node1"], r["node2"]))) in allowed_pairs,axis=1)]
pls_cpx['node1'] = pls_cpx['node1'].astype(str) + '-C'
pls_cpx['node2'] = pls_cpx['node2'].astype(str) + '-P'
pls_cpx.to_csv("PLS-CPX_updated.csv", index=False)
print("PLS-CPX",len(pls_cpx))


# CPX-CPX
cpx_cpx = pd.read_csv("matt_table.csv")
cpx_cpx = cpx_cpx.rename(columns={'Node 1 Name': 'node1','Node 2 Name': 'node2'})
cpx_cpx = cpx_cpx[(cpx_cpx["Node 1 Tissue"] == "Choroid Plexus") &(cpx_cpx["Node 2 Tissue"] == "Choroid Plexus")]
cpx_cpx = cpx_cpx[['node1', 'node2']]
cpx_cpx["node_min"] = cpx_cpx[["node1", "node2"]].min(axis=1)
cpx_cpx["node_max"] = cpx_cpx[["node1", "node2"]].max(axis=1)
cpx_cpx = cpx_cpx.drop_duplicates(subset=["node_min", "node_max"])
cpx_cpx = cpx_cpx.drop(columns=["node_min", "node_max"])
cpx_cpx = cpx_cpx.apply(lambda c: c.str.strip() if c.dtype == "object" else c)
cpx_cpx[['node1', 'node2']] = cpx_cpx[['node1', 'node2']].astype(str) + '-C'
cpx_cpx.to_csv("CPX-CPX_updated.csv", index=False)
print("CPX-CPX",len(cpx_cpx))



# CPX-CTX
cpx_ctx = pd.read_csv("matt_table.csv")
cpx_ctx = cpx_ctx.rename(columns={'Node 1 Name': 'node1','Node 2 Name': 'node2'})
cpx_ctx = cpx_ctx[(cpx_ctx["Node 1 Tissue"] == "Choroid Plexus") &(cpx_ctx["Node 2 Tissue"] == "Cortex")]
cpx_ctx = cpx_ctx[['node1', 'node2']]
cpx_ctx = cpx_ctx.apply(lambda c: c.str.strip() if c.dtype == "object" else c)
pls_cpx['node1'] = pls_cpx['node1'].astype(str) + '-C'
pls_cpx['node2'] = pls_cpx['node2'].astype(str) + '-T'
cpx_ctx.to_csv("CPX-CTX_updated.csv", index=False)
print("CPX-CTX",len(cpx_ctx))

# CTX-CTX
ctx_ctx = pd.read_csv("matt_table.csv")
ctx_ctx = ctx_ctx.rename(columns={'Node 1 Name': 'node1','Node 2 Name': 'node2'})
ctx_ctx = ctx_ctx[(ctx_ctx["Node 1 Tissue"] == "Cortex") &(ctx_ctx["Node 2 Tissue"] == "Cortex")]
ctx_ctx = ctx_ctx[['node1', 'node2']]
ctx_ctx["node_min"] = ctx_ctx[["node1", "node2"]].min(axis=1)
ctx_ctx["node_max"] = ctx_ctx[["node1", "node2"]].max(axis=1)
ctx_ctx = ctx_ctx.drop_duplicates(subset=["node_min", "node_max"])
ctx_ctx = ctx_ctx.drop(columns=["node_min", "node_max"])
ctx_ctx = ctx_ctx.apply(lambda c: c.str.strip() if c.dtype == "object" else c)
cpx_cpx[['node1', 'node2']] = cpx_cpx[['node1', 'node2']].astype(str) + '-T'
ctx_ctx.to_csv("CTX-CTX_updated.csv", index=False)
print("CTX-CTX",len(ctx_ctx))

print("Total",len(feci_feci)+len(feci_pls)+len(pls_pls)+len(pls_cpx)+len(cpx_cpx)+len(cpx_ctx)+len(ctx_ctx))

Feci-FECI 385
Feci-CPX 1666
Feci-PLS 1537
PLS-PLS 2200
PLS-CPX 668


/var/folders/0q/ndw_qsrd5cbch1pd347wrwx40000gn/T/ipykernel_6101/4013610926.py:90: DtypeWarning: Columns (43,45,56,102) have mixed types. Specify dtype option on import or set low_memory=False.
  cpx_cpx = pd.read_csv("matt_table.csv")


CPX-CPX 4069


/var/folders/0q/ndw_qsrd5cbch1pd347wrwx40000gn/T/ipykernel_6101/4013610926.py:106: DtypeWarning: Columns (43,45,56,102) have mixed types. Specify dtype option on import or set low_memory=False.
  cpx_ctx = pd.read_csv("matt_table.csv")


CPX-CTX 1092
CTX-CTX 2570
Total 12521


/var/folders/0q/ndw_qsrd5cbch1pd347wrwx40000gn/T/ipykernel_6101/4013610926.py:117: DtypeWarning: Columns (43,45,56,102) have mixed types. Specify dtype option on import or set low_memory=False.
  ctx_ctx = pd.read_csv("matt_table.csv")


In [4]:
import pandas as pd

files = [
    "Feci-Feci_updated.csv",
    "Feci-PLS_updated.csv",
    "PLS-PLS_updated.csv",
    "PLS-CPX_updated.csv",
    "CPX-CPX_updated.csv",
    "CPX-CTX_updated.csv",
    "CTX-CTX_updated.csv"
]

combined = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)

# strip whitespace from ALL string cells
combined = combined.apply(
    lambda col: col.str.strip() if col.dtype == "object" else col
)

print("Total edges", len(combined))
unique_nodes = pd.unique(combined[["node1", "node2"]].values.ravel())
print("Number of unique nodes:", len(unique_nodes))


combined.to_csv("./BiBC_Tables/final_edges_table.csv", index=False)

Total edges 12521
Number of unique nodes: 2763


In [7]:
import pandas as pd

node_type = pd.DataFrame(columns=["Node", "Type"])


# Feci-Feci
df_ff = pd.read_csv("Feci-Feci_updated.csv")
node_type = pd.concat([
    node_type,
    pd.DataFrame({"Node": df_ff["node1"], "Type": "Feci"}),
    pd.DataFrame({"Node": df_ff["node2"], "Type": "Feci"})
], ignore_index=True)
#print("Feci-FECI",len(df_ff))
#print(len(node_type))


# Feci-PLS
df_fp = pd.read_csv("Feci-PLS_updated.csv")
node_type = pd.concat([
    node_type,
    pd.DataFrame({"Node": df_fp["node1"], "Type": "Feci"}),
    pd.DataFrame({"Node": df_fp["node2"], "Type": "PLS"})
], ignore_index=True)
#print("Feci-PLS",len(node_type)-temp)
#print(len(node_type))


# PLS-PLS
df_pp = pd.read_csv("PLS-PLS_updated.csv")
node_type = pd.concat([
    node_type,
    pd.DataFrame({"Node": df_pp["node1"], "Type": "PLS"}),
    pd.DataFrame({"Node": df_pp["node2"], "Type": "PLS"})
], ignore_index=True)
#print(len(node_type))
#print("PLS-PLS",len(df_pp))
#print("Feci-PLS",len(node_type)-temp)


# PLS-CPX
df_pc = pd.read_csv("PLS-CPX_updated.csv")
node_type = pd.concat([
    node_type,
    pd.DataFrame({"Node": df_pc["node1"], "Type": "CPX"}),
    pd.DataFrame({"Node": df_pc["node2"], "Type": "PLS"})
], ignore_index=True)
#print("PLS-CPX",len(node_type)-temp)
#print(len(node_type))


# CPX-CPX
df_cc = pd.read_csv("CPX-CPX_updated.csv")
node_type = pd.concat([
    node_type,
    pd.DataFrame({"Node": df_cc["node1"], "Type": "CPX"}),
    pd.DataFrame({"Node": df_cc["node2"], "Type": "CPX"})
], ignore_index=True)
#print("CPX-CPX",len(node_type)-temp)
#print(len(node_type))


# CPX-CTX
df_ct = pd.read_csv("CPX-CTX_updated.csv")
node_type = pd.concat([
    node_type,
    pd.DataFrame({"Node": df_ct["node1"], "Type": "CPX"}),
    pd.DataFrame({"Node": df_ct["node2"], "Type": "CTX"})
], ignore_index=True)
#print("CPX-CTX",len(node_type)-temp)
#print(len(node_type))



# CTX-CTX
df_tt = pd.read_csv("CTX-CTX_updated.csv")
node_type = pd.concat([
    node_type,
    pd.DataFrame({"Node": df_tt["node1"], "Type": "CTX"}),
    pd.DataFrame({"Node": df_tt["node2"], "Type": "CTX"})
], ignore_index=True)
print(len(node_type))




node_type = node_type.map(lambda x: x.strip() if isinstance(x, str) else x)
node_type = node_type.drop_duplicates().sort_values("Node").reset_index(drop=True)

node_type.to_csv("./BiBC_Tables/ALL_node_types.csv", index=False, header=False)



# ----------------------------------------------------------------------------------


25042


In [9]:
import pandas as pd

# Read data
edges = pd.read_csv("./BiBC_Tables/final_edges_table.csv")
node_types = pd.read_csv(
    "./BiBC_Tables/ALL_node_types.csv",
    names=["Node", "Type"]
)

# Look according to Node and Type
type_map = dict(zip(node_types["Node"], node_types["Type"]))

# Map node types accoridng to type
edges["type1"] = edges["node1"].map(type_map)
edges["type2"] = edges["node2"].map(type_map)


# Save to file
typed_edges = edges[["node1", "type1", "node2", "type2"]]
typed_edges.to_csv(
    "./BiBC_Tables/test.csv",
    index=False
)
# ------------------------------------------------------------------------------------------
